# GIN Model for oral bioavailability dataset 

1. This notebook focuses on building a GIN model for oral bioavailabilty dataset using the GIN convolutional technique which is available on Pytorch Geometric 
2. The hyperparameters were found using the TPE algorithm using Optuna library after 30 trials 
3. Train/validate/test using 5-fold CV and process was repeated for 10 times.

### Note
1. Ensure that this notebook is in the same working directory as data folder, config.py, utils.py, engine.py and model.py
2. To load models, please download the saved models provided in google drive link from README.md
3. Comment away the training function to load saved models and reproduce results

In [2]:
from utils import seed_everything, LoadHOBDataset
from config import SEED_NO, NUM_FEATURES, NUM_GRAPHS_PER_BATCH, NUM_TARGET, DEVICE, PATIENCE, EPOCHS, N_SPLITS, params_gin
from engine import EngineHOB_no_edge
from model import GIN

import torch
import numpy as np
import optuna
from sklearn.model_selection import KFold
from torch_geometric.loader import DataLoader
import os 

## 2. Tuning of model (Suggest to skip and used the hyperparameters saved in config.py)
1. To tune the model, ensure data folder and this notebook are in the same working directory. Then, we make use of the Optuna library which allows us to use the Tree-structure Parzen Estimator Algorithm to find the best hyperparameters for us 
2. First, create a run_tuning function to include the training and validation step with early stopping mechansim 
3. Then, create an objective function of Optuna to evaluate the best hyperparameters in 30 trials. 

In [3]:
def run_tuning(train_loader, valid_loader, params):
    '''
    This function controls the tuning step of the model.

    Args:
    train_loader: Pytorch geometric DataLoader Class of train dataset
    valid_loader: Pytorch geometric DataLoader Class of validation dataset 
    params (dict): dictionary containing the hyperparameters (num_layers, hidden_size and learning_rate)
    '''
    model = GIN(num_features=NUM_FEATURES, num_targets=NUM_TARGET, num_layers=params['num_layers'], hidden_size=params['hidden_size'])
    model.to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr = params['learning_rate'])
    eng = EngineHOB_no_edge(model, optimizer, device=DEVICE)

    best_loss = np.inf
    early_stopping_iter = PATIENCE
    early_stopping_counter = 0 

    for epoch in range(EPOCHS):
        train_loss = eng.train(train_loader)
        valid_loss = eng.validate(valid_loader)
        print(f'Epoch: {epoch+1}/{EPOCHS}, train loss : {train_loss}, validation loss : {valid_loss}')
        if valid_loss < best_loss:
            best_loss = valid_loss 
            early_stopping_counter=0 #reset counter
        else:
            early_stopping_counter +=1

        if early_stopping_counter > early_stopping_iter:
            print('Early stopping...')
            break
        print(f'Early stop counter: {early_stopping_counter}')
    
    return best_loss

In [4]:
def objective(trial):
    params = {
        'num_layers' : trial.suggest_int('num_layers', 1,3),
        'hidden_size' : trial.suggest_int('hidden_size', 64, 512),
        'learning_rate' : trial.suggest_float('learning_rate', 1e-3, 9e-3, log=True)
    }

    #load dataset 
    dataset_for_cv = LoadHOBDataset(root='./data/graph_data/data_oral_avail_train/', raw_filename='data_oral_avail_train_50.csv')
    kf = KFold(n_splits=N_SPLITS)
    fold_loss = 0

    for fold_no, (train_idx, valid_idx) in enumerate(kf.split(dataset_for_cv)):
        print(f'Fold {fold_no}')
        train_dataset= []
        valid_dataset = []
        for t_idx in train_idx:
            train_dataset.append(torch.load(f'./data/graph_data/data_oral_avail_train/processed/molecule_{t_idx}.pt'))
        for v_idx in valid_idx:
            valid_dataset.append(torch.load(f'./data/graph_data/data_oral_avail_train/processed/molecule_{v_idx}.pt'))

        seed_everything(SEED_NO)
        train_loader = DataLoader(train_dataset, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)

        loss = run_tuning(train_loader, valid_loader, params)
        fold_loss += loss

    return fold_loss/5

study = optuna.create_study(direction = 'minimize')
study.optimize(objective, n_trials=30)
print(f'best trial:')
trial_ = study.best_trial
print(trial_.values)
print(f'Best parameters: {trial_.params}')

[I 2024-06-08 11:04:51,629] A new study created in memory with name: no-name-12e0106f-0861-4839-ad43-e226b5e7f9eb
Processing...
100%|██████████| 1157/1157 [00:20<00:00, 57.64it/s]
Done!


Fold 0
Epoch: 1/300, train loss : 10.258104115724564, validation loss : 0.6593297123908997
Early stop counter: 0
Epoch: 2/300, train loss : 0.9082609117031097, validation loss : 0.7059064507484436
Early stop counter: 1
Epoch: 3/300, train loss : 0.7551043778657913, validation loss : 0.6589779257774353
Early stop counter: 0
Epoch: 4/300, train loss : 0.6923689395189285, validation loss : 0.7521993517875671
Early stop counter: 1
Epoch: 5/300, train loss : 0.7061501443386078, validation loss : 0.7317740321159363
Early stop counter: 2
Epoch: 6/300, train loss : 0.692395880818367, validation loss : 0.6955903172492981
Early stop counter: 3
Epoch: 7/300, train loss : 0.6897939890623093, validation loss : 0.6833672523498535
Early stop counter: 4
Epoch: 8/300, train loss : 0.6894361227750778, validation loss : 0.6841514110565186
Early stop counter: 5
Epoch: 9/300, train loss : 0.684098020195961, validation loss : 0.6942744851112366
Early stop counter: 6
Epoch: 10/300, train loss : 0.68267497420

[I 2024-06-08 11:06:30,241] Trial 0 finished with value: 0.6533251643180847 and parameters: {'num_layers': 1, 'hidden_size': 360, 'learning_rate': 0.0064704715553700765}. Best is trial 0 with value: 0.6533251643180847.


Epoch: 15/300, train loss : 0.629514753818512, validation loss : 0.7236457467079163
Early stopping...
Fold 0
Epoch: 1/300, train loss : 5.876005202531815, validation loss : 0.67329341173172
Early stop counter: 0
Epoch: 2/300, train loss : 0.7045747488737106, validation loss : 0.6556058526039124
Early stop counter: 0
Epoch: 3/300, train loss : 0.7164165079593658, validation loss : 0.703113853931427
Early stop counter: 1
Epoch: 4/300, train loss : 0.6932381987571716, validation loss : 0.7209124565124512
Early stop counter: 2
Epoch: 5/300, train loss : 0.6911138594150543, validation loss : 0.6930795907974243
Early stop counter: 3
Epoch: 6/300, train loss : 0.6888187080621719, validation loss : 0.7041542530059814
Early stop counter: 4
Epoch: 7/300, train loss : 0.6887776851654053, validation loss : 0.6915023326873779
Early stop counter: 5
Epoch: 8/300, train loss : 0.6860926896333694, validation loss : 0.6850915551185608
Early stop counter: 6
Epoch: 9/300, train loss : 0.6889780610799789, 

[I 2024-06-08 11:07:10,735] Trial 1 finished with value: 0.6580754280090332 and parameters: {'num_layers': 3, 'hidden_size': 278, 'learning_rate': 0.003912478620054911}. Best is trial 0 with value: 0.6533251643180847.


...

Epoch: 13/300, train loss : 0.6603872627019882, validation loss : 0.6789872050285339
Early stopping...
Fold 0
Epoch: 1/300, train loss : 0.7880414724349976, validation loss : 0.6595366597175598
Early stop counter: 0
Epoch: 2/300, train loss : 0.6942510604858398, validation loss : 0.7183914184570312
Early stop counter: 1
Epoch: 3/300, train loss : 0.6871613264083862, validation loss : 0.7092657685279846
Early stop counter: 2
Epoch: 4/300, train loss : 0.6823095679283142, validation loss : 0.6720127463340759
Early stop counter: 3
Epoch: 5/300, train loss : 0.6834228336811066, validation loss : 0.6766470670700073
Early stop counter: 4
Epoch: 6/300, train loss : 0.6788733750581741, validation loss : 0.6957881450653076
Early stop counter: 5
Epoch: 7/300, train loss : 0.6842917948961258, validation loss : 0.6828497648239136
Early stop counter: 6
Epoch: 8/300, train loss : 0.677553653717041, validation loss : 0.686893880367279
Early stop counter: 7
Epoch: 9/300, train loss : 0.680360436439514

[I 2024-06-08 11:09:06,004] Trial 4 finished with value: 0.6606052041053772 and parameters: {'num_layers': 3, 'hidden_size': 139, 'learning_rate': 0.002709487733793013}. Best is trial 3 with value: 0.6423984169960022.


Epoch: 13/300, train loss : 0.6580397188663483, validation loss : 0.698117733001709
Early stopping...
Fold 0
Epoch: 1/300, train loss : 5.889975875616074, validation loss : 0.6361780762672424
Early stop counter: 0
Epoch: 2/300, train loss : 0.8004712760448456, validation loss : 0.671356737613678
Early stop counter: 1
Epoch: 3/300, train loss : 0.7255955636501312, validation loss : 0.7514684200286865
Early stop counter: 2
Epoch: 4/300, train loss : 0.7141314297914505, validation loss : 0.6931464076042175
Early stop counter: 3
Epoch: 5/300, train loss : 0.6926490515470505, validation loss : 0.6610605120658875
Early stop counter: 4
Epoch: 6/300, train loss : 0.6830406785011292, validation loss : 0.723013699054718
Early stop counter: 5
Epoch: 7/300, train loss : 0.6899322122335434, validation loss : 0.6803433895111084
Early stop counter: 6
Epoch: 8/300, train loss : 0.6864270269870758, validation loss : 0.6780078411102295
Early stop counter: 7
Epoch: 9/300, train loss : 0.6877624690532684,

[W 2024-06-08 11:09:20,377] Trial 5 failed with parameters: {'num_layers': 1, 'hidden_size': 502, 'learning_rate': 0.002589681050152906} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/donghai/miniconda3/envs/Vertical-GNN/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_781859/2398641158.py", line 26, in objective
    loss = run_tuning(train_loader, valid_loader, params)
  File "/tmp/ipykernel_781859/1548854241.py", line 20, in run_tuning
    train_loss = eng.train(train_loader)
  File "/mnt/d/Users/lenovo/Desktop/research_project/ADMET_property_prediction_model_validation/models/Hob-Pred-using-Transfer-Learning-main/engine.py", line 197, in train
    outputs = self.model(data.x, data.edge_index, data.batch)
  File "/home/donghai/miniconda3/envs/Vertical-GNN/lib/python3.9/site-packages/torch/nn/modules/module.py", line 1130, in _call_impl
    ret

KeyboardInterrupt: 

## 3. Train/validate/test

1. After hyperparameters tuning, the best parameters saved to config.py as params_gin
2. Next, we will train/validate/test the model using 5-fold CV followed by repeating the process for 5 times.
3. Ensure that data folder, from_scratch_trained_models folder and this notebook are in the same working directory
4. run_training to repeat the training process, please create a new folder to retrain, save and load saved model
5. Else, comment the run_training function and do testing to get the results which were obtained in the paper. 

In [2]:
def run_training(train_loader, valid_loader, params, trained_model_path):
    model = GIN(num_features=NUM_FEATURES, num_targets=NUM_TARGET, num_layers=params['num_layers'], hidden_size=params['hidden_size'])
    model.to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr = params['learning_rate'])
    eng = EngineHOB_no_edge(model, optimizer, device=DEVICE)

    best_loss = np.inf
    early_stopping_iter = PATIENCE
    early_stopping_counter = 0 

    for epoch in range(EPOCHS):
        train_loss= eng.train(train_loader)
        valid_loss= eng.validate(valid_loader)
        print(f'Epoch: {epoch+1}/{EPOCHS}, train loss : {train_loss}, validation loss : {valid_loss}')
        if valid_loss < best_loss:
            best_loss = valid_loss 
            early_stopping_counter=0 #reset counter
            print('Saving model...')
            torch.save(model.state_dict(), trained_model_path)
        else:
            early_stopping_counter +=1

        if early_stopping_counter > early_stopping_iter:
            print('Early stopping...')
            break
        print(f'Early stop counter: {early_stopping_counter}')
    
    return best_loss

def run_testing(test_loader, params, trained_model_path):
    model = GIN(num_features=NUM_FEATURES, num_targets=NUM_TARGET, num_layers=params['num_layers'], hidden_size=params['hidden_size'])
    model.load_state_dict(torch.load(trained_model_path))
    model.to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr = params['learning_rate'])
    eng = EngineHOB_no_edge(model, optimizer, device=DEVICE)

    print('Begin testing...')
    bce, acc, f1, roc_auc= eng.test(test_loader)
    print('Test completed!')
    print(f'bce:{bce}, acc :{acc}, f1: {f1}, roc_auc: {roc_auc}')
    return bce, acc, f1, roc_auc

In [7]:
n_repetitions = 5
params = params_gin
train_data_root_path = './data/graph_data/data_oral_avail_train/'
train_data_raw_filename = 'data_oral_avail_train_50.csv'
test_data_root_path = './data/graph_data/data_oral_avail_test/'
test_data_raw_filename = 'data_oral_avail_test_1_50.csv'
path_to_save_trained_model = './from_scratch_trained_models/GIN/'

bce_list = []
acc_list = []
f1_list = []
roc_auc_list = []

#load dataset 
dataset_for_cv = LoadHOBDataset(root=train_data_root_path, raw_filename=train_data_raw_filename)
test_dataset = LoadHOBDataset(root=test_data_root_path, raw_filename=test_data_raw_filename)

kf = KFold(n_splits= N_SPLITS)

for repeat in range(n_repetitions):
    for fold_no, (train_idx, valid_idx) in enumerate(kf.split(dataset_for_cv)):
        print(f'For rep: {repeat}, fold: {fold_no}')
        seed_everything(SEED_NO)
        train_dataset= []
        valid_dataset = []
        for t_idx in train_idx:
            train_dataset.append(torch.load(f'./data/graph_data/data_oral_avail_train/processed/molecule_{t_idx}.pt'))
        for v_idx in valid_idx:
            valid_dataset.append(torch.load(f'./data/graph_data/data_oral_avail_train/processed/molecule_{v_idx}.pt'))

        train_loader = DataLoader(train_dataset, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)

        run_training(train_loader, valid_loader, params, os.path.join(path_to_save_trained_model, f'gin_repeat_{repeat}_fold_{fold_no}.pt'))
        bce, acc, f1, roc_auc = run_testing(test_loader, params, os.path.join(path_to_save_trained_model, f'gin_repeat_{repeat}_fold_{fold_no}.pt'))
        bce_list.append(bce)
        acc_list.append(acc)
        f1_list.append(f1)
        roc_auc_list.append(roc_auc)

bce_arr = np.array(bce_list)
mean_bce = np.mean(bce_arr)
sd_bce = np.std(bce_arr)
print(f'bce:{mean_bce:.3f}±{sd_bce:.3f}')

acc_arr = np.array(acc_list)
acc_mean= np.mean(acc_arr)
acc_sd = np.std(acc_arr)
print(f'acc:{acc_mean:.3f}±{acc_sd:.3f}')

f1_arr = np.array(f1_list)
f1_mean= np.mean(f1_arr)
f1_sd = np.std(f1_arr)
print(f'f1: {f1_mean:.3f}±{f1_sd:.3f}')

roc_auc_arr = np.array(roc_auc_list)
roc_auc_mean= np.mean(roc_auc_arr)
roc_auc_sd = np.std(roc_auc_arr)
print(f'roc_auc: {roc_auc_mean:.3f}±{roc_auc_sd:.3f}')

For rep: 0, fold: 0
Epoch: 1/300, train loss : 1.4474536180496216, validation loss : 0.6468539237976074
Saving model...
Early stop counter: 0
Epoch: 2/300, train loss : 0.7944114208221436, validation loss : 0.7610831260681152
Early stop counter: 1
Epoch: 3/300, train loss : 0.7047408521175385, validation loss : 0.7292219996452332
Early stop counter: 2
Epoch: 4/300, train loss : 0.6896027475595474, validation loss : 0.7106878161430359
Early stop counter: 3
Epoch: 5/300, train loss : 0.686515599489212, validation loss : 0.6983362436294556
Early stop counter: 4
Epoch: 6/300, train loss : 0.6845705062150955, validation loss : 0.6978377103805542
Early stop counter: 5
Epoch: 7/300, train loss : 0.6809114813804626, validation loss : 0.7046172022819519
Early stop counter: 6
Epoch: 8/300, train loss : 0.6812890619039536, validation loss : 0.6997275948524475
Early stop counter: 7
Epoch: 9/300, train loss : 0.6773758679628372, validation loss : 0.6876591444015503
Early stop counter: 8
Epoch: 10/3